In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, BooleanType, DateType
from datetime import datetime

SOURCE_PATH     = "/Volumes/bfsi/landing/merchants/"
TABLE_NAME      = "bfsi.bronze_layer.merchants"
SCHEMA          = "bronze_layer"
CHECKPOINT_PATH = f"/Volumes/bfsi/landing/merchants/_checkpoint/{SCHEMA}_merchants"
BATCH_ID        = dbutils.widgets.get("batch_id") if "batch_id" in [w.name for w in dbutils.widgets.getAll()] else f"BATCH_{datetime.now().strftime('%Y%m%d_%H%M%S')}"

In [0]:
merchant_schema = StructType([
    StructField("merchant_id", StringType(), False),
    StructField("merchant_name", StringType(), True),
    StructField("merchant_category_code", StringType(), True),
    StructField("mcc_description", StringType(), True),
    StructField("risk_tier", IntegerType(), True),        # 1–5
    StructField("country", StringType(), True),
    StructField("is_blacklisted", BooleanType(), True),
    StructField("registration_date", DateType(), True)
])


In [0]:
df_merchants_stream = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "csv")
    .option("cloudFiles.includeExistingFiles", "true")
    .option("header", "true")
    .schema(merchant_schema)
    .load(SOURCE_PATH)
    .withColumn("_source_system", F.lit("MERCHANT_SYSTEM"))
    .withColumn("_load_ts", F.current_timestamp())
    .withColumn("_file_name", F.col("_metadata.file_path"))
    .withColumn("_batch_id", F.lit(BATCH_ID))
    .drop("_metadata")
    )

In [0]:
query = (
    df_merchants_stream.writeStream
    .format("delta")
    .option("checkpointLocation", CHECKPOINT_PATH)
    .outputMode("append")
    .trigger(availableNow=True)
    .toTable(TABLE_NAME)
)

query.awaitTermination()

print(f"Auto Loader stream completed for {TABLE_NAME}")

In [0]:
df_verify = spark.table(TABLE_NAME)

print(f"Total merchants ingested: {df_verify.count()}")
display(df_verify.limit(5))